In [13]:
import os
import re
import json
import time
import random
import datetime
import numpy as np
import pandas as pd
from tqdm import tqdm
from sklearn.model_selection import train_test_split


import torch
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import (
    AutoTokenizer,
    AutoModel,
    get_cosine_schedule_with_warmup,
)

from sklearn.metrics import accuracy_score
import evaluate

from peft import LoraConfig, get_peft_model

from transformers import AutoModelForCausalLM


In [14]:
torch.set_default_dtype(torch.float32)


In [15]:
f1_metric = evaluate.load("f1")

def format_time(elapsed):
    elapsed_rounded = int(round(elapsed))
    return str(datetime.timedelta(seconds=elapsed_rounded))


def remove_label_pattern(text):
    text = re.sub(
        r"(\[?\s*Justification\s*\]?:?\s*)|(\[Label\]:\s*(True|False|Conflicting))",
        "",
        text,
        flags=re.IGNORECASE,
    ).strip()
    return text.replace("\n", " ")


def print_trainable_parameters(model):
    trainable_params, all_param = 0, 0
    for _, param in model.named_parameters():
        all_param += param.numel()
        if param.requires_grad:
            trainable_params += param.numel()

    print(
        f"trainable params: {trainable_params} || "
        f"all params: {all_param} || "
        f"trainable%: {100 * trainable_params / all_param:.2f}"
    )
    
class FocalLoss(torch.nn.Module):
    def __init__(self, gamma=2.0, weight=None, label_smoothing=0.1):
        super().__init__()
        self.gamma = gamma
        self.register_buffer("weight", weight)
        self.label_smoothing = label_smoothing

    def forward(self, logits, targets):
        ce = torch.nn.functional.cross_entropy(
            logits, targets,
            weight=self.weight,
            label_smoothing=self.label_smoothing,
            reduction="none",
        )
        pt = torch.exp(-ce)
        return ((1.0 - pt) ** self.gamma * ce).mean()



In [16]:
class CustomClassifier(torch.nn.Module):
    def __init__(
        self,
        model_name,
        num_labels=1,
        hidden_dim=None,
        dropout_value=0.1,
        freeze_base_layer=True,
        use_lora=False,
        is_base_encoder=True,
        lora_rank=8,
        lora_alpha=16,
    ):
        super().__init__()

        self.is_base_encoder = is_base_encoder  # ← missing

        causal_lm = AutoModelForCausalLM.from_pretrained(
            model_name,
            dtype=torch.bfloat16,
            device_map="cuda",
        )
        self.model = causal_lm.model

        if freeze_base_layer:
            for param in self.model.parameters():
                param.requires_grad = False

        if use_lora:
            lora_config = LoraConfig(
                r=lora_rank,
                lora_alpha=lora_alpha,
                target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                                "gate_proj", "up_proj", "down_proj"],
                lora_dropout=0.1,
                bias="none",
            )
            self.model = get_peft_model(self.model, lora_config)

        hidden_size = self.model.config.hidden_size

        # AFTER:
        if hidden_dim:
            self.classifier = torch.nn.Sequential(
                torch.nn.Linear(hidden_size, 256),
                torch.nn.BatchNorm1d(256),        # ← add this line
                torch.nn.GELU(),
                torch.nn.Dropout(dropout_value),
                torch.nn.Linear(256, num_labels),
            )
        else:
            self.classifier = torch.nn.Linear(hidden_size, num_labels)

        self._init_weights(self.classifier)

    def _init_weights(self, module):
        for m in (module.modules() if isinstance(module, torch.nn.Sequential) else [module]):
            if isinstance(m, torch.nn.Linear):
                torch.nn.init.xavier_uniform_(m.weight)
                if m.bias is not None:
                    torch.nn.init.zeros_(m.bias)

    def forward(self, input_ids, attention_mask):
        outputs = self.model(input_ids=input_ids, attention_mask=attention_mask)

        if self.is_base_encoder:
            pooled_output = outputs.pooler_output
        else:
            pooled_output = outputs.last_hidden_state[:, -1, :]

        pooled_output = pooled_output.to(torch.device("cuda"), dtype=torch.bfloat16)  # retype this manually
        logits = self.classifier.to(torch.bfloat16)(pooled_output)
        return logits

In [17]:
class TextDataset(Dataset):
    def __init__(self, dataframe, tokenizer, max_length):
        self.texts  = dataframe["input_text"].tolist()
        self.labels = dataframe["Class"].tolist()
        self.tokenizer = tokenizer
        self.tokenizer.pad_token = self.tokenizer.eos_token
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx],
            truncation=True,
            padding="max_length",
            max_length=self.max_length,
            return_tensors="pt",
        )
        item = {k: v.squeeze(0) for k, v in encoding.items()}
        item["labels"] = torch.tensor(self.labels[idx], dtype=torch.long)  # ← float → long
        return item


In [18]:
class TrainerModule:
    def __init__(self, model, train_loader, val_loader, epochs, lr, patience, output_dir, train_df):
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.model = model.to(self.device)
        self.train_loader = train_loader
        self.val_loader = val_loader
        self.epochs = epochs
        self.optimizer =AdamW(model.parameters(), lr=lr, eps=1e-8, weight_decay=0.01)
        total_steps = len(train_loader) * epochs
        warmup_steps = int(0.05 * total_steps)
        self.scheduler = get_cosine_schedule_with_warmup(self.optimizer, warmup_steps, total_steps)
        self.output_dir = output_dir
        os.makedirs(output_dir, exist_ok=True)
        self.loss_fn = torch.nn.CrossEntropyLoss() 

    def train(self):
        for epoch in range(self.epochs):
            print(f"\nEpoch {epoch+1}/{self.epochs}")
            self.model.train()

            total_loss, total_acc = 0, 0

            for batch in tqdm(self.train_loader):
                self.optimizer.zero_grad()

                input_ids = batch["input_ids"].to(self.device)
                attention_mask = batch["attention_mask"].to(self.device)
                labels = batch["labels"].to(self.device)
                logits = self.model(input_ids, attention_mask)

                loss = self.loss_fn(logits.squeeze(1), labels)

                loss.backward()
                torch.nn.utils.clip_grad_norm_(self.model.parameters(), max_norm=1.0)
                self.optimizer.step()
                self.scheduler.step()

                total_loss += loss.item()

                preds = logits.argmax(dim=-1).cpu().numpy()
                total_acc += accuracy_score(labels.cpu().numpy(), preds)

            print(f"Train Loss: {total_loss / len(self.train_loader):.4f}")
            print(f"Train Acc: {total_acc / len(self.train_loader):.4f}")

            self.evaluate(epoch)

    def evaluate(self, epoch):
        self.model.eval()
        total_loss, total_acc = 0, 0

        with torch.no_grad():
            for batch in self.val_loader:
                input_ids = batch["input_ids"].to(self.device)
                attention_mask = batch["attention_mask"].to(self.device)
                labels = batch["labels"].to(self.device)

                logits = self.model(input_ids, attention_mask)
                loss = self.loss_fn(logits.squeeze(1), labels)

                total_loss += loss.item()
                preds = logits.argmax(dim=-1).cpu().numpy()
                total_acc += accuracy_score(labels.cpu().numpy(), preds)

        print(f"Val Loss: {total_loss / len(self.val_loader):.4f}")
        print(f"Val Acc: {total_acc / len(self.val_loader):.4f}")
        tokenizer.save_pretrained(self.output_dir)
        torch.save(
            self.model.state_dict(),
            os.path.join(self.output_dir, f"model_epoch_strategy5_8_QWEN_{epoch}"),
        )
        
    def _predictions(self, logits):
        return logits.argmax(dim=-1).cpu().numpy()   # ← was sigmoid >= 0.5


In [19]:
class VerifierEvaluator:
    def __init__(self, model_path, tokenizer_path, base_model, use_decomp, device="cuda"):
        self.device = torch.device(device if torch.cuda.is_available() else "cpu")
        self.use_decomp = use_decomp

        self.tokenizer = AutoTokenizer.from_pretrained(tokenizer_path)
        self.tokenizer.pad_token = self.tokenizer.eos_token

        # Infer num_labels from the checkpoint so the model always matches
        state_dict = torch.load(model_path, map_location=self.device)
        num_labels = state_dict["classifier.weight"].shape[0]  # e.g. 3

        self.model = CustomClassifier(
            BASE_MODEL,
            num_labels=2,
            use_lora=True,
            is_base_encoder=False,
#             hidden_dim=True,
        )
        self.model.load_state_dict(state_dict)
        self.model.to(self.device)
        self.model.eval()
        
    def encode_input(self, claim, questions, verdict, justification, max_length=1024):

        text = f"Claim: {claim}\nVerdict: {verdict}\nJustification: {justification}"

        encoding = self.tokenizer(
            text,
            truncation=True,
            padding="max_length",
            max_length=max_length,
            return_tensors="pt",
        )

        return (
            encoding["input_ids"].to(self.device),
            encoding["attention_mask"].to(self.device),
        )

    def score(self, claim, questions, verdict, justification):
        ids, mask = self.encode_input(claim, questions, verdict, justification)
        with torch.no_grad():
            logits = self.model(ids, mask)                        # (1, 3)
            probs  = torch.softmax(logits, dim=-1)               # (1, 3)
            return float(probs[0, 1].item())   # prob of class 1


In [20]:
print(torch.cuda.is_available())

True


In [ ]:
# ===== PATH PLACEHOLDERS =====
TRAIN_CSV = ""
VAL_CSV   = ""
TEST_CSV   = ""

OUTPUT_DIR = ""

BASE_MODEL = "Qwen/Qwen2.5-Math-7B"

MAX_LENGTH = 1024
BATCH_SIZE = 10
EPOCHS = 20
LR = 2e-5


In [22]:
from huggingface_hub import login
train_df = pd.read_json(TRAIN_CSV, lines = True)

In [23]:
train_df.head()

,sample_id,input_text,Label,Verdict,Class
0,0_a,Claim: “The first randomized controlled trial ...,conflicting,false,0
1,0_b,Claim: “The first randomized controlled trial ...,conflicting,false,0
2,0_c,Claim: “The first randomized controlled trial ...,conflicting,false,0
3,0_d,Claim: “The first randomized controlled trial ...,conflicting,false,0
4,0_e,Claim: “The first randomized controlled trial ...,conflicting,false,0


In [24]:
print(train_df["Class"].value_counts())

Class
0    22775
1     8658
Name: count, dtype: int64


In [25]:
train_df, val_df = train_test_split(
    train_df,
    test_size=0.2,
    stratify=train_df["Class"],
    random_state=42
)

In [26]:
!nvidia-smi

Tue May  5 20:17:00 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.105.08             Driver Version: 580.105.08     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA H200                    On  |   00000000:BB:00.0 Off |                    0 |
| N/A   34C    P0             75W /  700W |       4MiB / 143771MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [28]:
import torch
print(torch.__version__)

2.11.0+cu130


In [29]:

# val_df = pd.read_json(VAL_CSV, lines = True)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

train_dataset = TextDataset(train_df, tokenizer, MAX_LENGTH)
val_dataset = TextDataset(val_df, tokenizer, MAX_LENGTH)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE)

model = CustomClassifier(
    BASE_MODEL,
    num_labels=2,
    use_lora=True,
    is_base_encoder=False,
#     hidden_dim=True,
)

print_trainable_parameters(model)

trainer = TrainerModule(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    epochs=EPOCHS,
    lr=LR,
    patience=2,
    output_dir=OUTPUT_DIR,
    train_df=train_df,   # ← only new argument
)

trainer.train()


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

trainable params: 20192258 || all params: 7090811394 || trainable%: 0.28

Epoch 1/20


100%|██████████| 2515/2515 [21:07<00:00,  1.98it/s]


Train Loss: 1.1555
Train Acc: 0.6732
Val Loss: 0.5065
Val Acc: 0.7921

Epoch 2/20


100%|██████████| 2515/2515 [21:03<00:00,  1.99it/s]


...

Train Loss: 0.0226
Train Acc: 0.9937
Val Loss: 0.7818
Val Acc: 0.9149

Epoch 8/20


100%|██████████| 2515/2515 [21:01<00:00,  1.99it/s]


Train Loss: 0.0135
Train Acc: 0.9965
Val Loss: 0.8970
Val Acc: 0.9180

Epoch 9/20


 64%|██████▍   | 1621/2515 [13:33<07:28,  1.99it/s]


KeyboardInterrupt: 

In [ ]:
with open(TEST_CSV, "r") as f:
    test_data = json.load(f)

predictions = []

evaluator = VerifierEvaluator(
    model_path=f"model_epoch_strategy5_8_QWEN_5",
    tokenizer_path=BASE_MODEL,
    base_model=BASE_MODEL,
    use_decomp=True,
)

for idx, sample in enumerate(test_data):
    verdict_list = []
    verifier_score_list = []
    justification_list = []
    approved_majority_list = []

    for decoding_sample in range(1, len(sample["Reasoning_traces"]) + 1):
        justification = (
            remove_label_pattern(
                sample["Reasoning_traces"][decoding_sample - 1]
            ).split("Label:")[0]
        )
        score = evaluator.score(
            sample["claim"],
            sample.get("Questions", ""),
            sample["Verdict_list"][decoding_sample - 1].lower(),
            justification,
        )
        verdict_list.append(sample["Verdict_list"][decoding_sample - 1])
        justification_list.append(justification)
        verifier_score_list.append(score)

    best_verdict = verdict_list[np.argmax(np.array(verifier_score_list))]
    predictions.append({
        "query_id": idx,
        "Claim": sample["claim"],
        "Verdict_BoN": best_verdict,
        "BoN_Verdict_list": verdict_list,
        "Reasoning_traces": justification_list,
        "score_list": verifier_score_list,
    })

with open(f"Englisg_Test_Approach5_8_QWEN_1.json", "w") as fp:
    json.dump(predictions, fp, indent=4)

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]